# Jina v4 embeddings — Colab lub Mac

**GPU (T4):** Runtime → Change runtime type → **T4 GPU** (~12–14 h)

**Brak GPU w Colab?** Dwie opcje:
1. **Lepiej:** pobierz checkpoint z Drive → wznów na **Macu** (komórka na dole notebooka)
2. **Albo:** Runtime → **CPU** — działa, ale wolno (~20 h) i Colab często się rozłącza

| Krok | Gdzie | Czas |
|------|-------|------|
| embed-export | Colab GPU / Mac | 12–20 h |
| build-index | Mac | ~15 min |

In [ ]:
# === KONFIGURACJA ===
USE_DRIVE = True   # checkpoint na Drive (ZALECANE)
EMBED_BATCH_GPU = 16   # T4 GPU
EMBED_BATCH_CPU = 4    # gdy brak GPU (Runtime → CPU)

HF_TOKEN = ""  # opcjonalnie: "hf_..."

In [ ]:
import os
import zipfile
from google.colab import files

WORK_DIR = "/content/RAG"

if not os.path.isdir(os.path.join(WORK_DIR, "confluence")):
    print("Wybierz plik RAG-colab.zip z Maca...")
    uploaded = files.upload()
    zname = next(iter(uploaded))
    with zipfile.ZipFile(zname, "r") as zf:
        zf.extractall("/content")
    if not os.path.isdir(WORK_DIR):
        raise FileNotFoundError(
            "Po rozpakowaniu brak /content/RAG. "
            "Zip musi zawierać folder RAG/ z confluence/ i pipline.py"
        )

os.chdir(WORK_DIR)
print("OK — folder:", os.getcwd())
print("pliki confluence:", len(os.listdir("confluence")))

In [ ]:
DRIVE_OUT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = "/content/drive/MyDrive/RAG-embeddings"
    os.makedirs(DRIVE_OUT, exist_ok=True)
    os.makedirs(os.path.join(DRIVE_OUT, "build_checkpoint"), exist_ok=True)
    print("Checkpointy na Drive:", DRIVE_OUT)

In [ ]:
!pip install -q "transformers>=4.52,<5" sentence-transformers peft torchvision faiss-cpu rank_bm25 chonkie python-dotenv tqdm

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("Brak GPU — jedziemy na CPU (wolno ~20 h).")
    print("Lepiej: pobierz checkpoint z Drive i wznów na Macu (ostatnia komórka).")

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["EMBED_DEVICE"] = DEVICE
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if DRIVE_OUT:
    os.environ["EMBED_CHECKPOINT_DIR"] = os.path.join(DRIVE_OUT, "build_checkpoint")

import pipline
pipline.EMBED_BATCH_SIZE = EMBED_BATCH_GPU if DEVICE == "cuda" else EMBED_BATCH_CPU
print("Device:", DEVICE, "| batch:", pipline.EMBED_BATCH_SIZE)
print("Model:", pipline.EMBED_MODEL_NAME, "dim=", pipline.EMBED_DIM)

In [ ]:
# === GŁÓWNY KROK: embedowanie ~2–4 h ===
# Jak Colab się rozłączy, uruchom TĘ komórkę ponownie (wznowi z checkpointu na Drive).
pipline.embed_export_for_cloud()

In [ ]:
import shutil
from google.colab import files

emb = "faiss_index/embeddings.npy"
chk = "faiss_index/chunks_export.pkl"
assert os.path.exists(emb), "Brak embeddings.npy — embedowanie nie skończone"
assert os.path.exists(chk), "Brak chunks_export.pkl"

if DRIVE_OUT:
    shutil.copy2(emb, os.path.join(DRIVE_OUT, "embeddings.npy"))
    shutil.copy2(chk, os.path.join(DRIVE_OUT, "chunks_export.pkl"))
    print("Kopia na Drive:", DRIVE_OUT)

print("Pobieranie na Mac...")
files.download(emb)
files.download(chk)
print("\nNa Macu:")
print("  mkdir -p faiss_index")
print("  # wrzuć oba pliki do faiss_index/")
print("  python pipline.py build-index")

## Brak GPU w Colab? Wznów na Macu (polecane)

1. Na [drive.google.com](https://drive.google.com) → **RAG-embeddings/build_checkpoint/**
2. Pobierz: `embeddings.npy` + `progress.json`
3. Na Macu wrzuć do: `RAG/faiss_index/build_checkpoint/`
4. W terminalu:
```bash
cd /Users/karinaleskiewicz/RAG
/Users/karinaleskiewicz/iteracje_funkcji/venv/bin/python pipline.py embed-export
```
5. Szukaj: `Resuming embeddings from chunk ...` (~10% już zrobione)
6. Na końcu: `python pipline.py build-index`